## To-dos
0. grid search on baseline for getting optimal param space
1. Raw dataset loading
2. baseline training
3. get uncertain rows saved as a csv.
4. rerun the pipeline by uploading the full csv.
5. have a csv merging code in the beginning
6. all params including costs to be set in cell 1Rerun the pipeline by uploading the full CSV.

## Grid search

In [ ]:
!pip install optuna

In [ ]:
# ============================================================
#  XGBoost Fraud Detection — Optuna Tuner
#  Run this single cell. Edit the CONFIG block at the top.
# ============================================================

# ── INSTALL DEPENDENCIES (run once, then comment out) ───────
# !pip install xgboost optuna scikit-learn pandas numpy

# Assuming all labels are in a single column (1 = fraud, 0 = not fraud) and the insured amount are in a separate column.
# We also assume all labels for training and val data are correct as per ground truth and labels column has no empty cells.

# ╔══════════════════════════════════════════════════════════╗
# ║                    USER CONFIG                           ║
# ╚══════════════════════════════════════════════════════════╝

CSV_PATH            = "/content/synthetic_insurance_claims_with_fraud_5%_label.csv"   # path to your CSV file
TARGET_COL          = "fraud_label"     # column name: 1 = fraud, 0 = not fraud
SUM_INSURED_COL     = "sum_insured"     # column name holding the insured amount per row

COST_TP             = 100              # model says fraud (1), actually fraud (1)     → investigation cost
COST_FP             = 150              # model says fraud (1), actually not fraud (0) → false alarm cost + reputation damage
COST_TN             = 0               # model says not fraud (0), actually not fraud  → no cost (fixed)
# COST_FN           = 0.9 * sum_insured  per row (missed fraud = pay out 90% of claim)

TEST_SIZE           = 0.15            # fraction of data held out as final test set (never seen during tuning)
VAL_SIZE            = 0.15            # fraction of data held out as Optuna validation set (early stopping only)
                                      # remaining ~70% is used for CV inside Optuna
N_TRIALS            = 50              # number of Optuna search trials (more = better, slower)
N_FOLDS             = 5               # cross-validation folds (stratified by fraud label)
EARLY_STOPPING      = 50              # stop boosting if val business cost doesn't improve for this many rounds
N_WARMUP_STEPS      = 10              # trials before Optuna starts pruning weak trials
RANDOM_STATE        = 42
OUTPUT_JSON         = "best_params.json"  # where to save the best hyperparameters


# ╔══════════════════════════════════════════════════════════╗
# ║                    IMPORTS                               ║
# ╚══════════════════════════════════════════════════════════╝

import json
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ╔══════════════════════════════════════════════════════════╗
# ║             STEP 1 — LOAD, VALIDATE & SPLIT DATA         ║
# ╚══════════════════════════════════════════════════════════╝

print("[1/4] Loading data …")

df = pd.read_csv(CSV_PATH)

# ── sanity checks ────────────────────────────────────────────
for col in [TARGET_COL, SUM_INSURED_COL]:
    assert col in df.columns, f"Column '{col}' not found. Available: {list(df.columns)}"

assert df[TARGET_COL].isin([0, 1]).all(), \
    f"'{TARGET_COL}' must contain only 0 and 1. Found: {df[TARGET_COL].unique()}"

# ── encode categorical columns so XGBoost can handle them ────
X_raw = df.drop(columns=[TARGET_COL, SUM_INSURED_COL])
y_all = df[TARGET_COL].values.astype(np.int32)
si_all = df[SUM_INSURED_COL].values.astype(np.float64)

for col in X_raw.select_dtypes(include=["object", "category"]).columns:
    le = LabelEncoder()
    X_raw[col] = le.fit_transform(X_raw[col].astype(str))

X_all = X_raw.copy()

# ── 1. carve out held-out TEST set (never used during tuning) ─
X_trainval, X_test, y_trainval, y_test, si_trainval, si_test = train_test_split(
    X_all, y_all, si_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE,
)

# ── 2. carve out VALIDATION set (used only for early stopping) ─
# Relative val fraction within trainval
val_frac_adjusted = VAL_SIZE / (1.0 - TEST_SIZE)
X_train, X_val, y_train, y_val, si_train, si_val = train_test_split(
    X_trainval, y_trainval, si_trainval,
    test_size=val_frac_adjusted,
    stratify=y_trainval,
    random_state=RANDOM_STATE,
)

# ── 3. the remaining X_train / y_train is used for CV inside Optuna ─
ratio = np.sum(y_train == 0) / np.sum(y_train == 1)  # class imbalance ratio (train only)

print(f"    Total rows   : {len(df):,}")
print(f"    Train (CV)   : {len(X_train):,} | Fraud {np.mean(y_train)*100:.1f}%")
print(f"    Val (ES)     : {len(X_val):,}   | Fraud {np.mean(y_val)*100:.1f}%")
print(f"    Test (final) : {len(X_test):,}  | Fraud {np.mean(y_test)*100:.1f}%")
print(f"    Features     : {X_train.shape[1]}")


# ╔══════════════════════════════════════════════════════════╗
# ║          STEP 2 — CUSTOM BUSINESS COST FUNCTION          ║
# ╚══════════════════════════════════════════════════════════╝

def compute_business_cost(y_true, y_pred_proba, sum_insured_vals,
                           threshold=0.5,
                           cost_tp=COST_TP,
                           cost_fp=COST_FP):
    """Lower is better. Returns total cost over the provided split."""
    y_pred = (y_pred_proba >= threshold).astype(int)
    cost = np.zeros(len(y_true), dtype=np.float64)

    tp_mask = (y_pred == 1) & (y_true == 1)
    cost[tp_mask] = cost_tp                                         # investigation cost

    fp_mask = (y_pred == 1) & (y_true == 0)
    cost[fp_mask] = cost_fp                                         # false alarm cost

    fn_mask = (y_pred == 0) & (y_true == 1)
    cost[fn_mask] = 0.9 * np.maximum(sum_insured_vals[fn_mask], 0) # missed fraud → pay 90% of claim

    # cost_tn = 0 (no entry needed)
    return float(np.sum(cost))


def make_sample_weights(y_train_fold, si_train_fold):
    """
    Weight each training sample by its expected cost contribution.
    Fraud samples  → weighted by the payout we'd miss (0.9 * sum_insured).
    Non-fraud      → weighted by the false-alarm cost (COST_FP).
    scale_pos_weight is NOT used alongside these weights to avoid double-counting.
    """
    weights = np.where(
        y_train_fold == 1,
        0.9 * np.maximum(si_train_fold, 0),
        COST_FP
    ).astype(np.float64)
    weights = np.maximum(weights, 1e-6)   # XGBoost requires strictly positive weights
    weights = weights / weights.mean()    # normalise so mean weight ≈ 1
    return weights


def business_cost_custom_metric(y_proba, dtrain, si_vals, threshold):
    """
    XGBoost custom metric callback.
    Returns (metric_name, value) — lower is better (maximize=False).
    Normalises by number of rows so folds of different sizes are comparable.
    """
    y_true = dtrain.get_label()
    cost = compute_business_cost(y_true, y_proba, si_vals, threshold=threshold)
    return "business_cost", cost / max(len(y_true), 1)


# ╔══════════════════════════════════════════════════════════╗
# ║          STEP 3 — OPTUNA OBJECTIVE FUNCTION              ║
# ╚══════════════════════════════════════════════════════════╝
#
#  Architecture:
#    • Outer loop  → N_FOLDS stratified CV on X_train / y_train
#    • Inner early-stopping → uses the *held-out* X_val / y_val (not the CV fold)
#      This ensures early stopping is consistent across folds and doesn't leak
#      information between CV folds.
#    • Pruning     → trial.report() called AFTER each complete fold so Optuna
#      prunes based on real fold costs, not partial accumulations.
# ╚══════════════════════════════════════════════════════════╝

def objective(trial):
    params = {
        "objective":         "binary:logistic",
        "tree_method":       "hist",
        "seed":              RANDOM_STATE,
        "max_depth":         trial.suggest_int("max_depth", 3, 9),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 10),
        "gamma":             trial.suggest_float("gamma", 0.0, 1.0),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 2.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        # scale_pos_weight intentionally omitted — sample weights handle class imbalance
    }
    n_estimators = trial.suggest_int("n_estimators", 200, 2000)
    threshold    = trial.suggest_float("threshold", 0.05, 0.95)

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    fold_costs = []

    # Pre-build the early-stopping DMatrix (same for all folds — consistent signal)
    sw_val_es = make_sample_weights(y_val, si_val)
    dval_es   = xgb.DMatrix(X_val, label=y_val, weight=sw_val_es)

    for fold, (tr_idx, cv_val_idx) in enumerate(cv.split(X_train, y_train)):

        X_tr  = X_train.iloc[tr_idx]
        y_tr  = y_train[tr_idx]
        si_tr = si_train[tr_idx]

        X_cv_val  = X_train.iloc[cv_val_idx]
        y_cv_val  = y_train[cv_val_idx]
        si_cv_val = si_train[cv_val_idx]

        sw_tr  = make_sample_weights(y_tr, si_tr)
        dtrain = xgb.DMatrix(X_tr,     label=y_tr,     weight=sw_tr)
        dcvval = xgb.DMatrix(X_cv_val, label=y_cv_val)

        # Custom metric closure for early-stopping eval set (held-out val)
        def es_metric(y_pred_raw, dtrain_inner, _si=si_val, _th=threshold):
            return business_cost_custom_metric(y_pred_raw, dtrain_inner, _si, _th)

        booster = xgb.train(
            params,
            dtrain,
            num_boost_round       = n_estimators,
            evals                 = [(dval_es, "val_es")],     # early stopping on held-out val
            custom_metric         = es_metric,
            early_stopping_rounds = EARLY_STOPPING,
            maximize              = False,                      # minimise business cost
            verbose_eval          = False,
        )

        best_round = booster.best_iteration + 1  # best_iteration is 0-indexed

        # Evaluate on the CV fold's own held-out slice (unbiased fold estimate)
        y_proba_cv = booster.predict(dcvval, iteration_range=(0, best_round))
        fold_cost  = compute_business_cost(
            y_true=y_cv_val,
            y_pred_proba=y_proba_cv,
            sum_insured_vals=si_cv_val,
            threshold=threshold,
        )
        fold_costs.append(fold_cost / len(y_cv_val))  # normalise per-row

        # ── Pruning: report AFTER full fold completes ────────────
        trial.report(float(np.mean(fold_costs)), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return float(np.mean(fold_costs))


# ╔══════════════════════════════════════════════════════════╗
# ║              STEP 4 — RUN OPTUNA                         ║
# ╚══════════════════════════════════════════════════════════╝

print(f"[2/4] Starting Optuna search — {N_TRIALS} trials, {N_FOLDS}-fold CV …")

study = optuna.create_study(
    direction = "minimize",
    pruner    = optuna.pruners.MedianPruner(n_warmup_steps=N_WARMUP_STEPS),
    sampler   = optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"[3/4] Optuna finished. Best avg cost/row (CV): {study.best_value:,.4f}")


# ╔══════════════════════════════════════════════════════════╗
# ║     STEP 5 — FINAL EVALUATION ON HELD-OUT TEST SET       ║
# ╚══════════════════════════════════════════════════════════╝

print("[4/4] Evaluating best params on held-out test set …")

best = study.best_params.copy()
threshold_best = best.pop("threshold")
n_est_best     = best.pop("n_estimators")

best_xgb_params = {
    **best,
    "objective":   "binary:logistic",
    "tree_method": "hist",
    "seed":        RANDOM_STATE,
}

# Retrain on full train split (X_train) with best params
sw_full = make_sample_weights(y_train, si_train)
dtrain_full = xgb.DMatrix(X_train, label=y_train, weight=sw_full)
dval_full   = xgb.DMatrix(X_val,   label=y_val)
dtest_dm    = xgb.DMatrix(X_test,  label=y_test)

sw_val_es = make_sample_weights(y_val, si_val)
dval_full_w = xgb.DMatrix(X_val, label=y_val, weight=sw_val_es)

def final_es_metric(y_pred_raw, dtrain_inner, _si=si_val, _th=threshold_best):
    return business_cost_custom_metric(y_pred_raw, dtrain_inner, _si, _th)

final_booster = xgb.train(
    best_xgb_params,
    dtrain_full,
    num_boost_round       = n_est_best,
    evals                 = [(dval_full_w, "val_es")],
    custom_metric         = final_es_metric,
    early_stopping_rounds = EARLY_STOPPING,
    maximize              = False,
    verbose_eval          = False,
)

best_round_final = final_booster.best_iteration + 1
y_proba_test     = final_booster.predict(dtest_dm, iteration_range=(0, best_round_final))

test_cost       = compute_business_cost(y_test, y_proba_test, si_test, threshold=threshold_best)
test_cost_per_row = test_cost / len(y_test)

y_pred_test = (y_proba_test >= threshold_best).astype(int)
tp = int(np.sum((y_pred_test == 1) & (y_test == 1)))
fp = int(np.sum((y_pred_test == 1) & (y_test == 0)))
tn = int(np.sum((y_pred_test == 0) & (y_test == 0)))
fn = int(np.sum((y_pred_test == 0) & (y_test == 1)))
precision = tp / max(tp + fp, 1)
recall    = tp / max(tp + fn, 1)
f1        = 2 * precision * recall / max(precision + recall, 1e-9)

print(f"\n── Final Test Set Results ─────────────────────────────")
print(f"  Threshold            : {threshold_best:.4f}")
print(f"  Best boost round     : {best_round_final}")
print(f"  Total business cost  : {test_cost:,.2f}")
print(f"  Cost per row         : {test_cost_per_row:,.4f}")
print(f"  TP / FP / TN / FN   : {tp} / {fp} / {tn} / {fn}")
print(f"  Precision            : {precision:.4f}")
print(f"  Recall               : {recall:.4f}")
print(f"  F1                   : {f1:.4f}")
print(f"───────────────────────────────────────────────────────\n")


# ╔══════════════════════════════════════════════════════════╗
# ║              STEP 6 — SAVE BEST PARAMS                   ║
# ╚══════════════════════════════════════════════════════════╝

best_params_out = study.best_params.copy()
best_params_out["_meta"] = {
    "best_avg_cost_per_row_cv"   : round(study.best_value, 4),
    "test_cost_per_row"          : round(test_cost_per_row, 4),
    "test_total_cost"            : round(test_cost, 4),
    "test_precision"             : round(precision, 4),
    "test_recall"                : round(recall, 4),
    "test_f1"                    : round(f1, 4),
    "test_tp_fp_tn_fn"           : [tp, fp, tn, fn],
    "n_trials_run"               : N_TRIALS,
    "n_folds"                    : N_FOLDS,
    "train_size"                 : len(X_train),
    "val_size"                   : len(X_val),
    "test_size"                  : len(X_test),
    "cost_tp"                    : COST_TP,
    "cost_fp"                    : COST_FP,
    "cost_fn_formula"            : "0.9 * sum_insured",
    "cost_tn"                    : 0,
    "csv_path"                   : CSV_PATH,
    "target_col"                 : TARGET_COL,
    "sum_insured_col"            : SUM_INSURED_COL,
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(best_params_out, f, indent=2)

print(f"Best params saved → '{OUTPUT_JSON}'")

[1/4] Loading data …
    Total rows   : 500,000
    Train (CV)   : 350,000 | Fraud 5.0%
    Val (ES)     : 75,000   | Fraud 5.0%
    Test (final) : 75,000  | Fraud 5.0%
    Features     : 43
[2/4] Starting Optuna search — 50 trials, 5-fold CV …


  0%|          | 0/50 [00:00<?, ?it/s]

[3/4] Optuna finished. Best avg cost/row (CV): 5.8031
[4/4] Evaluating best params on held-out test set …

── Final Test Set Results ─────────────────────────────
  Threshold            : 0.0729
  Best boost round     : 181
  Total business cost  : 569,921.00
  Cost per row         : 7.5989
  TP / FP / TN / FN   : 3749 / 324 / 70926 / 1
  Precision            : 0.9205
  Recall               : 0.9997
  F1                   : 0.9585
───────────────────────────────────────────────────────

Best params saved → 'best_params.json'


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv('/content/synthetic_insurance_claims_with_fraud_5%_label.csv')


# Total rows = 50,000. 15k is 30% of the data.
df_train, df_test = train_test_split(
    df.iloc[:95000],
    test_size=30000,
    random_state=42,
    stratify=df.iloc[:95000]['fraud_label'] # Highly recommended!
)

# Verification
print(f"Test shape: {df_test.shape}")
print(f"Train shape: {df_train.shape}")

# Checking if the fraud distribution stayed roughly the same
print(f"Frauds in train = {df_train['fraud_label'].sum() / len(df_train) : .4%}")
print(f"Frauds in test = {df_test['fraud_label'].sum() / len(df_test) : .4%}")

df_train.to_csv('labeled_data.csv', index=False)
df_train.to_csv('labeled_data_original.csv', index=False)
df_test.to_csv('unlabeled_data_with_fraud_label.csv', index=False)
df_test.drop(columns=['fraud_label']).to_csv('unlabeled_data.csv', index=False)

Test shape: (30000, 45)
Train shape: (65000, 45)
Frauds in train =  4.9985%
Frauds in test =  4.9967%


## AL begins

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║                    USER CONFIG                           ║
# ╚══════════════════════════════════════════════════════════╝

# ── File paths ───────────────────────────────────────────────
LABELED_CSV        = "labeled_data.csv"    # initial labeled training pool
UNLABELED_CSV      = "unlabeled_data.csv"  # pool of unlabeled rows to query
BEST_PARAMS_JSON   = "best_params.json"    # output of xgb_fraud_tuner_v2.ipynb

# ── Column names (must match both CSVs) ──────────────────────
TARGET_COL         = "fraud_label"         # label column (0 / 1) in labeled CSV
SUM_INSURED_COL    = "sum_insured"         # insured amount column

# ── Active Learning settings ─────────────────────────────────
N_AL_LOOPS         = 10                   # number of AL iterations to run
N_QUERY_PER_LOOP   = 100                  # uncertain points to query each loop
EARLY_STOPPING = 50                       # stop if business cost on val doesn't improve for this many rounds

# ── Output ───────────────────────────────────────────────────
UNCERTAIN_CSV_PREFIX = "uncertain_round"   # saved as uncertain_round_1.csv, etc.
FINAL_MODEL_PATH     = "al_final_model.json"  # final retrained XGBoost model
FINAL_NON_SEQUENTIAL_MODEL_PATH = "nonsequential_final_model.json"
updated_uncertain_csv_path = "updated_uncertain_points.csv"  # saved after each AL iteration with updated points.

# ── Business cost params (must match what the tuner used) ────
COST_TP            = 100
COST_FP            = 150

## Non sequential branch -- run just once

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings("ignore")


def load_params_from_json(json_path):
    """
    Loads best_params.json written by xgb_fraud_tuner_v2.
    Returns (xgb_params_dict, n_estimators, threshold).
    Strips 'threshold', 'n_estimators', and '_meta' — these are handled
    separately by xgb.train (num_boost_round) and the AL loop.
    """
    with open(json_path) as f:
        raw = json.load(f)

    params        = {k: v for k, v in raw.items() if not k.startswith("_")}
    threshold     = params.pop("threshold", 0.5)
    n_estimators  = params.pop("n_estimators", 300)   # used as num_boost_round

    return params, n_estimators, threshold

def build_and_train_model(params, n_estimators, threshold, X_train, y_train, si_train):
    """
    Trains XGBoost via xgb.train (identical pattern to xgb_fraud_tuner_v2).
    - sample_weight encodes the business cost asymmetry into every gradient step
    - custom_metric (business_cost_eval) drives early stopping — NOT logloss
    Returns a raw xgb.Booster.
    """
    sw     = make_sample_weights(y_train, si_train)
    dtrain = xgb.DMatrix(X_train, label=y_train, weight=sw)
    dval   = xgb.DMatrix(X_train, label=y_train)   # use train as val proxy when no holdout

    # Bind threshold into the eval closure (mirrors the fold_eval pattern in the tuner)
    def _cost_eval(y_proba, dtrain_inner, _si=si_train, _th=threshold):
        return business_cost_eval(y_proba, dtrain_inner, _si, _th)

    xgb_train_params = {
        "objective"   : "binary:logistic",
        "tree_method" : "hist",
        "seed"        : 42,
        "verbosity"   : 0,
        **params,
    }

    booster = xgb.train(
        xgb_train_params,
        dtrain,
        num_boost_round       = n_estimators,
        evals                 = [(dval, "train")],
        custom_metric         = _cost_eval,        # early stopping watches business cost
        early_stopping_rounds = EARLY_STOPPING,
        maximize              = False,
        verbose_eval          = False,
    )
    return booster


print("✅ Helpers for non-sequential defined.")

✅ Helpers for non-sequential defined.


In [ ]:
# ── Load unlabeled pool ───────────────────────────────────────
df_unlabeled = pd.read_csv(f'{UNLABELED_CSV[:-4]}_with_fraud_label.csv')

assert SUM_INSURED_COL in df_unlabeled.columns, \
    f"'{SUM_INSURED_COL}' must exist in {UNLABELED_CSV}"
# assert TARGET_COL not in df_unlabeled.columns, \
#     f"'{TARGET_COL}' found in unlabeled CSV — remove it first."

# Keep only feature cols + sum_insured; reset index for safe row-dropping
df_unlabeled = df_unlabeled.reset_index(drop=True)

print(f"[UNLABELED POOL] {len(df_unlabeled):,} rows loaded from '{UNLABELED_CSV}'")

# Randomly sample 1,000 points
df_sampled = df_unlabeled.sample(n=1000, random_state=42)

# Verify the shape
print(f"New shape: {df_sampled.shape}")
# ── Working copies (mutated each loop) ───────────────────────

# ── Load tuned hyper-parameters ──────────────────────────────
xgb_params, N_ESTIMATORS, THRESHOLD = load_params_from_json(BEST_PARAMS_JSON)

print(f"[BASELINE] Loaded params from '{BEST_PARAMS_JSON}'")
print(f"  Threshold : {THRESHOLD}")
print(f"  XGB params:")
for k, v in xgb_params.items():
    print(f"    {k:<25} = {v}")

# ── Load labeled training pool ────────────────────────────────
df_labeled = pd.read_csv(LABELED_CSV)

for col in [TARGET_COL, SUM_INSURED_COL]:
    assert col in df_labeled.columns, \
        f"Column '{col}' not found in {LABELED_CSV}. Available: {list(df_labeled.columns)}"

assert df_labeled[TARGET_COL].isin([0, 1]).all(), \
    f"'{TARGET_COL}' must contain only 0 and 1."

FEATURE_COLS = [c for c in df_labeled.columns if c not in [TARGET_COL, SUM_INSURED_COL]]

print("FEATURE_COLS = ", FEATURE_COLS)
X_pool = df_labeled[FEATURE_COLS].copy()
y_pool = df_labeled[TARGET_COL].values.astype(np.int32)
si_pool = df_labeled[SUM_INSURED_COL].values.astype(np.float64)

print(f"\n[LABELED POOL] {len(df_labeled):,} rows | "
      f"fraud={y_pool.sum():,} ({100*y_pool.mean():.1f}%) | "
      f"features={len(FEATURE_COLS)}")

# ── Train baseline model on full labeled pool ─────────────────
print("\n[BASELINE] Training baseline XGBoost on labeled pool …")
baseline_model = build_and_train_model(xgb_params, N_ESTIMATORS, THRESHOLD, X_pool, y_pool, si_pool)

current_model   = baseline_model           # starts as baseline (xgb.Booster), updated each loop
X_train_current = X_pool.copy()            # grows as new labels arrive
y_train_current = y_pool.copy()
si_train_current = si_pool.copy()
df_sampled_pool = df_sampled.copy()    # shrinks as rows are queried

[UNLABELED POOL] 30,000 rows loaded from 'unlabeled_data.csv'
New shape: (1000, 45)
[BASELINE] Loaded params from 'best_params.json'
  Threshold : 0.07287721406968567
  XGB params:
    max_depth                 = 9
    min_child_weight          = 5
    gamma                     = 0.1195942459383017
    learning_rate             = 0.09273146363606355
    subsample                 = 0.8803925243084487
    colsample_bytree          = 0.7367663185416977
    colsample_bylevel         = 0.8625803079727365
    reg_alpha                 = 0.0001256064834846832
    reg_lambda                = 0.0003525633816265316
FEATURE_COLS =  ['claim_days_differenc', 'claim_loss_date_dom', 'claim_loss_datet_to_policy_ply_icp_dt_delta', 'claim_loss_date_to_policy_str_date_delta', 'claim_loss_type_ab', 'claim_loss_type_ad', 'claim_loss_type_al', 'claim_loss_type_eo', 'claim_loss_type_es', 'claim_loss_type_ew', 'claim_loss_type_fi', 'claim_loss_type_fl', 'claim_loss_type_gm', 'claim_loss_type_im', 'claim_loss_

In [ ]:
# ── Expand training pool ─────────────────────────
X_new  = df_sampled_pool[FEATURE_COLS].values
y_new  = df_sampled_pool[TARGET_COL].values.astype(np.int32)
si_new = df_sampled_pool[SUM_INSURED_COL].values.astype(np.float64)

X_train_current  = pd.concat([X_train_current,
                                pd.DataFrame(X_new, columns=FEATURE_COLS)],
                                ignore_index=True)
y_train_current  = np.concatenate([y_train_current, y_new])
si_train_current = np.concatenate([si_train_current, si_new])

# ── Retrain model on expanded pool ───────────────
current_model = build_and_train_model(
    xgb_params, N_ESTIMATORS, THRESHOLD, X_train_current, y_train_current, si_train_current
)
train_proba = current_model.predict(
    xgb.DMatrix(X_train_current),
    iteration_range=(0, current_model.best_iteration + 1)
)
train_cost  = compute_business_cost(y_train_current, train_proba,
                                        si_train_current, THRESHOLD)
train_auc   = roc_auc_score(y_train_current, train_proba)

al_log = []
loop_stats = {
    "train_pool_size"   : len(y_train_current),
    "fraud_in_query"    : int(y_new.sum()),
    "train_auc"         : round(train_auc, 4),
    "train_business_cost": round(train_cost, 2),
}
al_log.append(loop_stats)
print(  f"train_pool={loop_stats['train_pool_size']:,}  "
        f"fraud_in_query={loop_stats['fraud_in_query']}  "
        f"AUC={loop_stats['train_auc']:.4f}  "
        f"biz_cost={loop_stats['train_business_cost']:,.0f}  ")

train_pool=68,000  fraud_in_query=46  AUC=1.0000  biz_cost=517,800  


In [ ]:
# ── Save final retrained XGBoost booster ─────────────────────
current_model.save_model(FINAL_NON_SEQUENTIAL_MODEL_PATH)
print(f"✅ Final AL model saved → '{FINAL_NON_SEQUENTIAL_MODEL_PATH}'")

# ── AL progression table ─────────────────────────────────────
df_log = pd.DataFrame(al_log)
print(f"\n{'─'*90}")
print(f"  Active Learning Progression Summary")
print(f"{'─'*90}")
print(df_log.to_string(index=False))
print(f"{'─'*90}")

# ── Delta vs baseline ─────────────────────────────────────────
final_auc  = df_log["train_auc"].iloc[-1]
final_cost = df_log["train_business_cost"].iloc[-1]

print(f"\n  Baseline → Final AL model (train-set, indicative only)")
print(f"  AUC          : {final_auc:.4f}  ")
# print(f"  AUC          : {baseline_auc:.4f}  →  {final_auc:.4f}  "
#       f"(Δ {final_auc - baseline_auc:+.4f})")
# print(f"  Business cost: {baseline_cost:,.0f}  →  {final_cost:,.0f}  "
#       f"(Δ {final_cost - baseline_cost:+,.0f})")

print(f"\n💡 Next step: evaluate '{FINAL_NON_SEQUENTIAL_MODEL_PATH}' on a held-out labeled test set.")
print(f"   Load with: booster = xgb.Booster(); "
      f"booster.load_model('{FINAL_NON_SEQUENTIAL_MODEL_PATH}')")

✅ Final AL model saved → 'nonsequential_final_model.json'

──────────────────────────────────────────────────────────────────────────────────────────
  Active Learning Progression Summary
──────────────────────────────────────────────────────────────────────────────────────────
 train_pool_size  fraud_in_query  train_auc  train_business_cost
           68000              46        1.0             517800.0
──────────────────────────────────────────────────────────────────────────────────────────

  Baseline → Final AL model (train-set, indicative only)
  AUC          : 1.0000  

💡 Next step: evaluate 'nonsequential_final_model.json' on a held-out labeled test set.
   Load with: booster = xgb.Booster(); booster.load_model('nonsequential_final_model.json')


## Sequential AL branch -- to be run 10 times

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings("ignore")

def compute_business_cost(y_true, y_pred_proba, sum_insured_vals,
                           threshold=0.5, cost_tp=COST_TP, cost_fp=COST_FP):
    """Identical cost function from xgb_fraud_tuner_v2 — lower is better."""
    y_pred             = (y_pred_proba >= threshold).astype(int)
    cost               = np.zeros(len(y_true), dtype=np.float64)
    cost[(y_pred==1) & (y_true==1)] = cost_tp
    cost[(y_pred==1) & (y_true==0)] = cost_fp
    cost[(y_pred==0) & (y_true==1)] = 0.9 * sum_insured_vals[(y_pred==0) & (y_true==1)]
    return float(np.sum(cost))


def make_sample_weights(y_train, sum_insured_train, cost_fp=COST_FP):
    """Per-row weights encoding business cost asymmetry into XGBoost training."""
    weights = np.where(
        y_train == 1,
        0.9 * sum_insured_train,   # fraud rows: weight = missed-fraud cost
        cost_fp                    # clean rows: weight = false-alarm cost
    )
    weights /= weights.mean()
    return weights.astype(np.float64)


def business_cost_eval(y_proba, dtrain, sum_insured_vals, threshold):
    """
    Custom eval metric for xgb.train.
    Called every boosting round so early stopping is driven by business cost,
    not logloss.
    y_proba is already a probability because objective='binary:logistic'.
    """
    y_true     = dtrain.get_label()
    cost       = compute_business_cost(y_true, y_proba, sum_insured_vals,
                                        threshold=threshold)
    normalised = cost / max(len(y_true), 1)   # per-row cost for comparability
    return "business_cost", normalised


def load_params_from_json(json_path):
    """
    Loads best_params.json written by xgb_fraud_tuner_v2.
    Returns (xgb_params_dict, n_estimators, threshold).
    Strips 'threshold', 'n_estimators', and '_meta' — these are handled
    separately by xgb.train (num_boost_round) and the AL loop.
    """
    with open(json_path) as f:
        raw = json.load(f)

    params        = {k: v for k, v in raw.items() if not k.startswith("_")}
    threshold     = params.pop("threshold", 0.5)
    n_estimators  = params.pop("n_estimators", 300)   # used as num_boost_round

    return params, n_estimators, threshold

def build_and_train_model(params, n_estimators, threshold, X_train, y_train, si_train):
    """
    Trains XGBoost via xgb.train (identical pattern to xgb_fraud_tuner_v2).
    - sample_weight encodes the business cost asymmetry into every gradient step
    - custom_metric (business_cost_eval) drives early stopping — NOT logloss
    Returns a raw xgb.Booster.
    """
    sw     = make_sample_weights(y_train, si_train)
    dtrain = xgb.DMatrix(X_train, label=y_train, weight=sw)
    dval   = xgb.DMatrix(X_train, label=y_train)   # use train as val proxy when no holdout

    # Bind threshold into the eval closure (mirrors the fold_eval pattern in the tuner)
    def _cost_eval(y_proba, dtrain_inner, _si=si_train, _th=threshold):
        return business_cost_eval(y_proba, dtrain_inner, _si, _th)

    xgb_train_params = {
        "objective"   : "binary:logistic",
        "tree_method" : "hist",
        "seed"        : 42,
        "verbosity"   : 0,
        **params,
    }

    booster = xgb.train(
        xgb_train_params,
        dtrain,
        num_boost_round       = n_estimators,
        evals                 = [(dval, "train")],
        custom_metric         = _cost_eval,        # early stopping watches business cost
        early_stopping_rounds = EARLY_STOPPING,
        maximize              = False,
        verbose_eval          = False,
    )
    return booster

print("✅ Helpers defined.")

✅ Helpers defined.


In [ ]:
# ── Load tuned hyper-parameters ──────────────────────────────
xgb_params, N_ESTIMATORS, THRESHOLD = load_params_from_json(BEST_PARAMS_JSON)

print(f"[BASELINE] Loaded params from '{BEST_PARAMS_JSON}'")
print(f"  Threshold : {THRESHOLD}")
print(f"  XGB params:")
for k, v in xgb_params.items():
    print(f"    {k:<25} = {v}")

# ── Load labeled training pool ────────────────────────────────
df_labeled = pd.read_csv(LABELED_CSV)

for col in [TARGET_COL, SUM_INSURED_COL]:
    assert col in df_labeled.columns, \
        f"Column '{col}' not found in {LABELED_CSV}. Available: {list(df_labeled.columns)}"

assert df_labeled[TARGET_COL].isin([0, 1]).all(), \
    f"'{TARGET_COL}' must contain only 0 and 1."

FEATURE_COLS = [c for c in df_labeled.columns if c not in [TARGET_COL, SUM_INSURED_COL]]

print("FEATURE_COLS = ", FEATURE_COLS)
X_pool = df_labeled[FEATURE_COLS].copy()
y_pool = df_labeled[TARGET_COL].values.astype(np.int32)
si_pool = df_labeled[SUM_INSURED_COL].values.astype(np.float64)

print(f"\n[LABELED POOL] {len(df_labeled):,} rows | "
      f"fraud={y_pool.sum():,} ({100*y_pool.mean():.1f}%) | "
      f"features={len(FEATURE_COLS)}")

# ── Train baseline model on full labeled pool ─────────────────
print("\n[BASELINE] Training baseline XGBoost on labeled pool …")
baseline_model = build_and_train_model(xgb_params, N_ESTIMATORS, THRESHOLD, X_pool, y_pool, si_pool)

baseline_proba = baseline_model.predict(xgb.DMatrix(X_pool),
                     iteration_range=(0, baseline_model.best_iteration + 1))
baseline_cost  = compute_business_cost(y_pool, baseline_proba, si_pool, THRESHOLD)
baseline_auc   = roc_auc_score(y_pool, baseline_proba)

print(f"  ✅ Baseline trained.")
print(f"  Baseline AUC (train)          : {baseline_auc:.4f}")
print(f"  Baseline business cost (train): {baseline_cost:,.0f}")
print(f"  (Train-set scores are optimistic — used only as AL progress reference)")
print(f'  Number of rows in LABELED_CSV = {len(df_labeled)}')

[BASELINE] Loaded params from 'best_params.json'
  Threshold : 0.07287721406968567
  XGB params:
    max_depth                 = 9
    min_child_weight          = 5
    gamma                     = 0.1195942459383017
    learning_rate             = 0.09273146363606355
    subsample                 = 0.8803925243084487
    colsample_bytree          = 0.7367663185416977
    colsample_bylevel         = 0.8625803079727365
    reg_alpha                 = 0.0001256064834846832
    reg_lambda                = 0.0003525633816265316
FEATURE_COLS =  ['claim_days_differenc', 'claim_loss_date_dom', 'claim_loss_datet_to_policy_ply_icp_dt_delta', 'claim_loss_date_to_policy_str_date_delta', 'claim_loss_type_ab', 'claim_loss_type_ad', 'claim_loss_type_al', 'claim_loss_type_eo', 'claim_loss_type_es', 'claim_loss_type_ew', 'claim_loss_type_fi', 'claim_loss_type_fl', 'claim_loss_type_gm', 'claim_loss_type_im', 'claim_loss_type_lb', 'claim_loss_type_mc', 'claim_loss_type_md', 'claim_loss_type_rl', 'claim_l

In [ ]:
import pandas as pd

temp = pd.read_csv(UNLABELED_CSV)
len(temp)

30000

In [ ]:
# ── Load unlabeled pool ───────────────────────────────────────
# df_unlabeled = pd.read_csv(UNLABELED_CSV)
# df_unlabeled = pd.read_csv('/content/unlabeled_csv_29900_rows.csv')
df_unlabeled = pd.read_csv('/content/unlabeled_csv_29800_rows.csv')

assert SUM_INSURED_COL in df_unlabeled.columns, \
    f"'{SUM_INSURED_COL}' must exist in {UNLABELED_CSV}"
assert TARGET_COL not in df_unlabeled.columns, \
    f"'{TARGET_COL}' found in unlabeled CSV — remove it first."

# Keep only feature cols + sum_insured; reset index for safe row-dropping
df_unlabeled = df_unlabeled.reset_index(drop=True)

# print(f"[UNLABELED POOL] {len(df_unlabeled):,} rows loaded from '{UNLABELED_CSV}'")
print(f"[UNLABELED POOL] {len(df_unlabeled):,} rows loaded from /content/unlabeled_csv_29800_rows.csv ")

# ── Working copies (mutated each loop) ───────────────────────
current_model   = baseline_model           # starts as baseline (xgb.Booster), updated each loop
X_train_current = X_pool.copy()            # grows as new labels arrive
y_train_current = y_pool.copy()
si_train_current = si_pool.copy()
df_unlabeled_pool = df_unlabeled.copy()    # shrinks as rows are queried

[UNLABELED POOL] 29,800 rows loaded from /content/unlabeled_csv_29800_rows.csv 


In [ ]:
# ── Track per-loop stats ──────────────────────────────────────
al_log = []

print(f"\n{'═'*65}")
print(f"  Starting Active Learning — {N_AL_LOOPS} loops × {N_QUERY_PER_LOOP} queries")
print(f"{'═'*65}\n")

print(f'  Number of rows in df_unlabeled_pool = {len(df_unlabeled_pool)}')
# ── STEP 1: Score unlabeled pool ─────────────────────────
X_unlabeled = df_unlabeled_pool[FEATURE_COLS]
proba_unlabeled = current_model.predict(
    xgb.DMatrix(X_unlabeled),
    iteration_range=(0, current_model.best_iteration + 1)
)    # returns probabilities directly (binary:logistic)
si_pool_vals = df_unlabeled_pool[SUM_INSURED_COL].values

# ── STEP 2: Select top-N most uncertain rows ─────────────
# -- cost_weighted: uncertainty scaled by insured amount --
uncertainty = 0.5 - np.abs(proba_unlabeled - 0.5)
norm_cost = si_pool_vals / (si_pool_vals.max() + 1)
scores = uncertainty * norm_cost
query_local_idx = np.argsort(-scores)[:N_QUERY_PER_LOOP]

# -- uncertainty: closest to decision boundary --
# uncertainty = np.abs(proba_unlabeled - 0.5)
# query_local_idx = np.argsort(uncertainty)[:N_QUERY_PER_LOOP]

# -- diversity: k-means cluster + uncertainty (requires: from sklearn.cluster import MiniBatchKMeans) --
# from sklearn.preprocessing import StandardScaler
# X_scaled = StandardScaler().fit_transform(X_unlabeled)
# kmeans = MiniBatchKMeans(n_clusters=min(N_QUERY_PER_LOOP, len(X_scaled)//2), random_state=42, batch_size=256).fit(X_scaled)
# dists = np.min(kmeans.transform(X_scaled), axis=1)
# scores = (1 - np.abs(2*proba_unlabeled - 1)) + 0.5*(dists/(dists.max()+1e-6))
# query_local_idx = np.argsort(-scores)[:N_QUERY_PER_LOOP]

# ── End strategies ────────────────────────────────────────

query_df          = df_unlabeled_pool.iloc[query_local_idx].copy()
query_proba       = proba_unlabeled[query_local_idx]
query_uncertainty = scores[query_local_idx]

query_df["al_pred_proba"]  = query_proba
query_df["al_uncertainty"] = query_uncertainty

# ── STEP 3: Save uncertain points CSV ────────────────────
uncertain_csv_path = f"{UNCERTAIN_CSV_PREFIX}.csv"
query_df.to_csv(uncertain_csv_path, index=False)


═════════════════════════════════════════════════════════════════
  Starting Active Learning — 10 loops × 100 queries
═════════════════════════════════════════════════════════════════

  Number of rows in df_unlabeled_pool = 29800


In [ ]:
import pandas as pd

# 1. Load the files
df_uncertain = pd.read_csv('/content/uncertain_round.csv')
df_unlabeled_with_labels = pd.read_csv('/content/unlabeled_data_with_fraud_label.csv')

# 2. Identify the common columns to join on
# We exclude 'fraud_label' because it only exists in one of the files
common_cols = [col for col in df_uncertain.columns if col in df_unlabeled_with_labels.columns]

# 3. Perform a left merge
# This finds the matching row in the labeled file for every row in the uncertain file
df_updated = pd.merge(
    df_uncertain,
    df_unlabeled_with_labels[common_cols + ['fraud_label']],
    on=common_cols,
    how='left'
)

# 4. Save the result
df_updated.to_csv('updated_uncertain_points.csv', index=False)

print(f"Successfully matched {len(df_updated)} rows with their fraud labels.")

Successfully matched 100 rows with their fraud labels.


In [ ]:
# ── STEP 4: Load back from CSV ────────────────────────────
#   (In production: fill the 'fraud_label' column with real oracle labels,
#    save the file, then let this cell load the annotated version.)

newly_labeled = pd.read_csv(updated_uncertain_csv_path)

# Merge the newly queried points (query_df) into the labeled pool
df_labeled_updated = pd.concat([df_labeled, newly_labeled], ignore_index=True)

# Save the updated labeled pool back to CSV
df_labeled_updated.drop(columns=['al_pred_proba', 'al_uncertainty']).to_csv(LABELED_CSV, index=False)
print(f"Added {len(newly_labeled)} points. New labeled pool size: {len(df_labeled_updated)}")


# ── STEP 5: Expand training pool ─────────────────────────
X_new  = newly_labeled[FEATURE_COLS].values
y_new  = newly_labeled[TARGET_COL].values.astype(np.int32)
si_new = newly_labeled[SUM_INSURED_COL].values.astype(np.float64)

X_train_current  = pd.concat([X_train_current,
                                pd.DataFrame(X_new, columns=FEATURE_COLS)],
                                ignore_index=True)
y_train_current  = np.concatenate([y_train_current, y_new])
si_train_current = np.concatenate([si_train_current, si_new])

# ── STEP 6: Remove queried rows from unlabeled pool ──────
pool_indices_to_drop = df_unlabeled_pool.index[query_local_idx]
df_unlabeled_pool    = df_unlabeled_pool.drop(index=pool_indices_to_drop).reset_index(drop=True)
df_unlabeled_pool.to_csv(f'unlabeled_csv_{len(df_unlabeled_pool)}_rows.csv')

# ── STEP 7: Retrain model on expanded pool ───────────────
current_model = build_and_train_model(
    xgb_params, N_ESTIMATORS, THRESHOLD, X_train_current, y_train_current, si_train_current
)

# ── STEP 8: Compute train-set stats for progress tracking ─
train_proba = current_model.predict(
    xgb.DMatrix(X_train_current),
    iteration_range=(0, current_model.best_iteration + 1)
)
train_cost  = compute_business_cost(y_train_current, train_proba,
                                        si_train_current, THRESHOLD)
train_auc   = roc_auc_score(y_train_current, train_proba)

loop_stats = {
    "train_pool_size"   : len(y_train_current),
    "unlabeled_remaining": len(df_unlabeled_pool),
    "fraud_in_query"    : int(y_new.sum()),
    "mean_uncertainty"  : round(float(query_uncertainty.mean()), 4),
    "train_auc"         : round(train_auc, 4),
    "train_business_cost": round(train_cost, 2),
    "uncertain_csv"     : uncertain_csv_path,
}
al_log.append(loop_stats)

print(  f"train_pool={loop_stats['train_pool_size']:,}  "
        f"unlabeled_left={loop_stats['unlabeled_remaining']:,}  "
        f"fraud_in_query={loop_stats['fraud_in_query']}  "
        f"mean_uncertainty={loop_stats['mean_uncertainty']:.4f}  "
        f"AUC={loop_stats['train_auc']:.4f}  "
        f"biz_cost={loop_stats['train_business_cost']:,.0f}  "
        f"→ saved {uncertain_csv_path}")

print(f"\n{'═'*65}")
print(f"  Active Learning iteration complete")
print(f"{'═'*65}")

Added 100 points. New labeled pool size: 65300
train_pool=65,300  unlabeled_left=29,700  fraud_in_query=2  mean_uncertainty=0.0994  AUC=1.0000  biz_cost=375,800  → saved uncertain_round.csv

═════════════════════════════════════════════════════════════════
  Active Learning iteration complete
═════════════════════════════════════════════════════════════════


In [ ]:
# ── Save final retrained XGBoost booster ─────────────────────
current_model.save_model(FINAL_MODEL_PATH)
print(f"✅ Final AL model saved → '{FINAL_MODEL_PATH}'")

# ── AL progression table ─────────────────────────────────────
df_log = pd.DataFrame(al_log)
print(f"\n{'─'*90}")
print(f"  Active Learning Progression Summary")
print(f"{'─'*90}")
print(df_log.to_string(index=False))
print(f"{'─'*90}")

# ── Delta vs baseline ─────────────────────────────────────────
final_auc  = df_log["train_auc"].iloc[-1]
final_cost = df_log["train_business_cost"].iloc[-1]

print(f"\n  Baseline → Final AL model (train-set, indicative only)")
print(f"  AUC          : {baseline_auc:.4f}  →  {final_auc:.4f}  "
      f"(Δ {final_auc - baseline_auc:+.4f})")
print(f"  Business cost: {baseline_cost:,.0f}  →  {final_cost:,.0f}  "
      f"(Δ {final_cost - baseline_cost:+,.0f})")

print(f"\n💡 Next step: evaluate '{FINAL_MODEL_PATH}' on a held-out labeled test set.")
print(f"   Load with: booster = xgb.Booster(); "
      f"booster.load_model('{FINAL_MODEL_PATH}')")

✅ Final AL model saved → 'al_final_model.json'

──────────────────────────────────────────────────────────────────────────────────────────
  Active Learning Progression Summary
──────────────────────────────────────────────────────────────────────────────────────────
 train_pool_size  unlabeled_remaining  fraud_in_query  mean_uncertainty  train_auc  train_business_cost       uncertain_csv
           65300                29700               2            0.0994        1.0             375800.0 uncertain_round.csv
──────────────────────────────────────────────────────────────────────────────────────────

  Baseline → Final AL model (train-set, indicative only)
  AUC          : 1.0000  →  1.0000  (Δ +0.0000)
  Business cost: 444,600  →  375,800  (Δ -68,800)

💡 Next step: evaluate 'al_final_model.json' on a held-out labeled test set.
   Load with: booster = xgb.Booster(); booster.load_model('al_final_model.json')
